# Microduck RL — Colab에서 공식 `microduck_rl` 학습

런타임 → 런타임 유형 변경 → **GPU** (무료 T4도 동작, L4/A100이면 훨씬 빠름).
체크포인트는 Google Drive(`MyDrive/microduck/logs`)에 저장되므로 세션이 끊겨도 이어서 학습할 수 있습니다.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Drive 연결 + 설치

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
os.environ['UV_HTTP_TIMEOUT'] = '600'

%cd /content
!test -d microduck_rl || git clone https://github.com/pollen-robotics/microduck_rl
%cd /content/microduck_rl
!git pull -q && git log -1 --oneline

# 로그(체크포인트)를 Drive에 두기
!mkdir -p /content/drive/MyDrive/microduck/logs
!rm -rf logs && ln -s /content/drive/MyDrive/microduck/logs logs

In [ ]:
!uv sync 2>&1 | tail -3   # CUDA 휠 포함 수 GB, 몇 분 걸림

In [ ]:
!uv run list-envs 2>/dev/null | grep MicroDuck | head -40

## 2. 학습

- `TASK`: 위 목록에서 고르기. 실제 로봇이 쓰는 걷기는 `Mjlab-VelStand-Rough-Backlash-MicroDuck` 계열.
- `NUM_ENVS`: 공식 4096. T4에서 메모리 부족이면 2048/1024로.
- `MAX_ITERS`: 반복 횟수. 공식 README 기준 4096 envs로 1~2시간이면 쓸 만한 걸음.

In [ ]:
TASK = 'Mjlab-Velocity-Flat-MicroDuck'
NUM_ENVS = 4096
MAX_ITERS = 3000

!uv run train {TASK} --env.scene.num-envs {NUM_ENVS} --agent.max-iterations {MAX_ITERS} --agent.logger tensorboard 2>&1 | grep -vE "^\s*$" | tail -n 40

### 이어서 학습 (세션이 끊긴 뒤)
가장 최근 run의 마지막 체크포인트에서 재개합니다.

In [ ]:
import glob, os, re
ckpts = sorted(glob.glob('logs/rsl_rl/*/*/model_*.pt'), key=os.path.getmtime)
last = ckpts[-1]; print(last)
run_dir = os.path.basename(os.path.dirname(last))
!uv run train {TASK} --env.scene.num-envs {NUM_ENVS} --agent.max-iterations {MAX_ITERS} --agent.logger tensorboard \
    --agent.resume True --agent.load-run {run_dir} --agent.load-checkpoint {os.path.basename(last)} 2>&1 | tail -n 40

## 3. 학습 곡선 (TensorBoard)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/rsl_rl

## 4. ONNX로 내보내기 → Mac으로 받기
관측 정규화가 그래프에 포함되는 공식 경로(`scripts/export.py`)만 사용하세요.

In [ ]:
import glob, os
last = sorted(glob.glob('logs/rsl_rl/*/*/model_*.pt'), key=os.path.getmtime)[-1]
print('checkpoint:', last)
!uv run scripts/export.py {TASK} --checkpoint-file {last} 2>&1 | tail -5
!find . -maxdepth 4 -name '*.onnx' -newer {last} -not -path './.venv/*' | head

In [ ]:
# 찾은 onnx를 Drive에 복사 (Mac의 Google Drive 폴더에서 바로 보임)
import glob, os, shutil
onnx = sorted([p for p in glob.glob('**/*.onnx', recursive=True) if '.venv' not in p], key=os.path.getmtime)[-1]
dst = f'/content/drive/MyDrive/microduck/{TASK}.onnx'
shutil.copy(onnx, dst); print(onnx, '->', dst)

## 5. Mac에서 확인

```sh
cd ~/Projects/microduck/microduck_rl
uv run scripts/infer_policy.py --walking ~/Downloads/Mjlab-Velocity-Flat-MicroDuck.onnx   # 키보드로 조종
```